In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.table("spotify.spotify_user_demo")
print((df.count(), len(df.columns)))

(108000, 6)


In [0]:
spotify_user_demo = spark.table("spotify.spotify_user_demo").toPandas()
spotify_user_behavior = spark.table("spotify.spotify_user_behavior").toPandas()


In [0]:
#3 SCHEMA + DATA TYPES
spotify_user_behavior.info()
spotify_user_demo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108000 entries, 0 to 107999
Data columns (total 26 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   user_id                           108000 non-null  int64  
 1   daily_listening_minutes           108000 non-null  object 
 2   sessions_per_day                  108000 non-null  int64  
 3   days_active_last_30               108000 non-null  object 
 4   avg_session_minutes               108000 non-null  object 
 5   skip_rate                         108000 non-null  float64
 6   liked_songs_pct                   108000 non-null  object 
 7   ads_skipped_pct                   108000 non-null  object 
 8   playlists_followed                108000 non-null  int64  
 9   artists_followed                  108000 non-null  int64  
 10  genre_diversity_score             108000 non-null  object 
 11  mean_danceability                 108000 non-null  o

In [0]:
#4 FILL RATE/ MISSINGNESS
def fill_rate(df: pd.DataFrame) -> pd.DataFrame:
    out = (
        df.isna()
        .mean()
        .mul(100)
        .round(2)
        .sort_values()
        .reset_index()
    )
    out.columns = ["column", "missing_pct"]
    out["fill_rate_pct"] = (100-out["missing_pct"]).round(2)
    return out
    

In [0]:
print("\n Fill rate: User behavior---")
print(fill_rate(spotify_user_behavior).to_string(index=False))



 Fill rate: User behavior---
                          column  missing_pct  fill_rate_pct
                         user_id          0.0          100.0
median_gap_minutes_between_plays          0.0          100.0
              repeat_artist_rate          0.0          100.0
               repeat_track_rate          0.0          100.0
                       std_tempo          0.0          100.0
                     std_valence          0.0          100.0
                      std_energy          0.0          100.0
                      mean_tempo          0.0          100.0
           mean_instrumentalness          0.0          100.0
                mean_speechiness          0.0          100.0
               mean_acousticness          0.0          100.0
                    mean_valence          0.0          100.0
                     mean_energy          0.0          100.0
               mean_danceability          0.0          100.0
           genre_diversity_score          0.0          

In [0]:
print("\n Fill rate: User DEMO---")
print(fill_rate(spotify_user_demo).to_string(index=False))


 Fill rate: User DEMO---
                    column  missing_pct  fill_rate_pct
                   user_id          0.0          100.0
                       age          0.0          100.0
                   country          0.0          100.0
                 city_tier          0.0          100.0
               device_type          0.0          100.0
subscription_tenure_months          0.0          100.0


In [0]:
#5 BASIC SANITY CHECKS
print("\n DESCRIBE : USER BEHAVIOR(numeric)")
print(spotify_user_behavior.drop(columns=["user_id"]).describe().round(3).to_string())

print("\n DESCRIBE : USER DEMO(numeric)")
demo_numeric = spotify_user_demo.select_dtypes(include=[np.number])
if demo_numeric.shape[1] > 0:
    print(demo_numeric.describe().T.round(3).to_string())
else:
    print("No numeric columns in demo besides user_id(if any).")


 DESCRIBE : USER BEHAVIOR(numeric)
       sessions_per_day   skip_rate  playlists_followed  artists_followed  repeat_track_rate
count        108000.000  108000.000          108000.000        108000.000         108000.000
mean              1.308       0.326               9.165            15.969              0.347
std               0.798       0.179              42.990            77.937              0.180
min               1.000       0.001               1.000             1.000              0.001
25%               1.000       0.187               1.000             1.000              0.207
50%               1.000       0.306               1.000             2.000              0.327
75%               1.000       0.442               3.000             4.000              0.468
max              15.000       1.000             500.000           800.000              1.000

 DESCRIBE : USER DEMO(numeric)
              count       mean        std  min       25%      50%       75%       max
user_id  

In [0]:
#5 BASIC SANITY CHECKS
print("\n DESCRIBE : USER BEHAVIOR(numeric)")
print(spotify_user_behavior.drop(columns=["user_id"]).describe().T.round(3).to_string())

print("\n DESCRIBE : USER DEMO(numeric)")
demo_numeric = spotify_user_demo.select_dtypes(include=[np.number])
if demo_numeric.shape[1] > 0:
    print(demo_numeric.describe().T.round(3).to_string())
else:
    print("No numeric columns in demo besides user_id(if any).")


 DESCRIBE : USER BEHAVIOR(numeric)
                       count    mean     std    min    25%    50%    75%    max
sessions_per_day    108000.0   1.308   0.798  1.000  1.000  1.000  1.000   15.0
skip_rate           108000.0   0.326   0.179  0.001  0.187  0.306  0.442    1.0
playlists_followed  108000.0   9.165  42.990  1.000  1.000  1.000  3.000  500.0
artists_followed    108000.0  15.969  77.937  1.000  1.000  2.000  4.000  800.0
repeat_track_rate   108000.0   0.347   0.180  0.001  0.207  0.327  0.468    1.0

 DESCRIBE : USER DEMO(numeric)
              count       mean        std  min       25%      50%       75%       max
user_id    108000.0  54000.500  31177.059  1.0  27000.75  54000.5  81000.25  108000.0
city_tier  108000.0      1.848      0.792  1.0      1.00      2.0      2.00       3.0


In [0]:
issues =[]
def add_issue(table,col,issue,count):
    issues.append({"table":table,"col":col,"issue":issue,"rows_affected": int(count)})

In [0]:
behavior_rules = {
    "daily_listening_minutes": (0,None),
    "sessions_per_day": (0,None),
    "days_active_last_30": (0,30),
    "avg_active_last_30": (0,None),

    "skip_rate": (0,1),
    "liked_songs_pct": (0,1),
    "ads_skipped_pct": (0,1),

    "genre_diversity_score": (0,1),
    "mean_danceability": (0,1),
    "mean_energy": (0,1),
    "mean_valence": (0,1),
    "mean_acousticness": (0,1),
    "mean_speechiness": (0,1),
    "mean_instrumentalness": (0,1),
    
    "mean_tempo": (0,None),
    "std_energy": (0,None),
    "std_valence": (0,None),
    "std_tempo": (0,None),

    "repeat_track_rate": (0,1),
    "repeat_artist_rate": (0,1),

    "median_gap_minutes_between_plays": (0,None),
    
    "mean_track_popularity": (0,1),
    "pct_top_popularity_tracks": (0,1),
}

In [0]:
for col,(lo,high) in behavior_rules.items():
    if col in spotify_user_behavior.columns:
        s = spotify_user_behavior[col]
        if lo is not None:
            bad = (s<lo).sum()
            if bad > 0:
                add_issue("spotify_user_behavior",col, f"value < {lo}", bad)
        if high is not None:
            bad = (s>high).sum()
            if bad > 0:
                add_issue("spotify_user_behavior",col, f"value > {high}", bad)

In [0]:
demo_rules = {
    "age": (0,120),
    "city_tier": (1,3),
    "subscription_tenure_months": (0,None),
}
for col,(lo,high) in demo_rules.items():
    if col in spotify_user_demo.columns and pd.api.types.is_numeric_dtype(spotify_user_demo[col]):
        s = spotify_user_demo[col]
        if lo is not None:
            bad = (s<lo).sum()
            if bad > 0:
                add_issue("spotify_user_demo",col, f"value < {lo}", bad)
        if high is not None:
            bad = (s>high).sum()
            if bad > 0:
                add_issue("spotify_user_demo",col, f"value > {high}", bad)

In [0]:
for col in ["country","device_type"]:
    if col in spotify_user_demo.columns:
        blank = (spotify_user_demo[col].astype(str).str.strip() == "").sum()
        if blank > 0:
            add_issue("spotify_user_demo",col, "blank string", blank)

In [0]:
print("\n SANITY CHECK ISSUES FOUND")
if len(issues) > 0 :
    print("No obvious rule based issues found")
else:
    issues_df = pd.DataFrame(issues).sort_values(["table","rows_affected"], ascending=[True,False])
    display(issues_df.to_string(index=False))


 SANITY CHECK ISSUES FOUND
No obvious rule based issues found
